# Building conversational applications with Strands Agents

In this notebook, you learn how to use Strands agents to integrate external capabilities into conversational applications.

This is a conversion of the original module9-converse_with_tools.ipynb that used the Converse API with tool specifications. Instead, we'll use Strands agents for a more streamlined approach.

## Environment setup

In this task, you set up your environment and install the required packages for Strands agents.

In [1]:
%pip install -U langchain-community

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\disha\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
from langchain_community.tools import DuckDuckGoSearchRun
from strands import Agent, tool
from strands.models.bedrock import BedrockModel
from strands_tools import calculator

model_id = "amazon.nova-lite-v1:0"

C:\Users\disha\AppData\Local\Temp\ipykernel_16504\845463919.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [1]:
%pip install strands-agents-tools

Defaulting to user installation because normal site-packages is not writeable
  Using cached aws_requests_auth-0.4.3-py2.py3-none-any.whl.metadata (567 bytes)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached python_multipart-0.0.32-py3-none-any.whl.metadata (2.1 kB)
  Using cached pywin32-312-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached truststore-0.10.4-py3-none-any.whl.metadata (4.4 kB)
  Using cached click-


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\disha\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Define Tools for Strands Agents

In this task, we define the tools that our Strands agent will use. Instead of manually handling tool specifications and execution like in the original notebook, Strands agents handle this automatically.

In [3]:
@tool
def my_calculator_tool(oper1: int, oper2: int, operator: str) -> str:
    """
    Perform basic arithmetic operations on two numbers.
    
    Args:
        oper1: First operand
        oper2: Second operand  
        operator: Operation to perform (+, -, *, / or add, subtract, multiply, divide)
    
    Returns:
        Result of the calculation as a string
    """
    def add(a, b):
        return a + b

    def subtract(a, b):
        return a - b

    def multiply(a, b):
        return a * b

    def divide(a, b):
        if b == 0:
            return "Cannot divide by zero"
        else:
            return float(a / b)
        
    if operator == '+' or operator == 'add':
        return str(add(oper1, oper2))
    elif operator == '-' or operator == 'subtract':
        return str(subtract(oper1, oper2))
    elif operator == '*' or operator == 'multiply':
        return str(multiply(float(oper1), float(oper2)))
    elif operator == '/' or operator == 'divide':
        return str(divide(oper1, oper2))
    else:
        return "Invalid operator"

In [4]:
@tool
def duckduckgo_search(query: str) -> str:
    """
    Search with DuckDuckGo for information about a topic.
    
    Args:
        query: The search query
    
    Returns:
        DuckDuckGo search results as a string
    """
    tool = DuckDuckGoSearchRun()
    result = tool.invoke(query)
    return result

## Create Strands Agent

Now we create a Strands agent that uses the Bedrock LLM and our defined tools. The agent will automatically handle tool calling and conversation flow.

In [5]:
# Create the Bedrock LLM wrapper for Strands
llm = BedrockModel(
    model_id=model_id
)

# Create the agent with tools
agent = Agent(
    model=llm,
    tools=[duckduckgo_search,my_calculator_tool],
    system_prompt="You are a helpful assistant that can perform calculations and search for information on DuckDuckGo. Use the available tools when needed to provide accurate and helpful responses."
)

## Test the Strands Agent

Now let's test our Strands agent with the same query used in the original notebook.

In [6]:
# Test the agent with a complex query that requires both tools
query = "What is Amazon SageMaker? What is launch year multiplied by 2"

print(f"Query: {query}\n")
print("Agent Response:")
print("=" * 50)

# Run the agent
response = agent(query)
print(response)

Query: What is Amazon SageMaker? What is launch year multiplied by 2

Agent Response:
<thinking>To answer this question, I need to provide information about Amazon SageMaker and perform a multiplication calculation on its launch year. I will first need to search for information about Amazon SageMaker, including its launch year, and then multiply that year by 2 using my calculator tool.</thinking>

Tool #1: duckduckgo_search
<thinking>The tool to search with DuckDuckGo encountered an error. Instead, I will provide general information about Amazon SageMaker and perform the calculation using the known launch year. Amazon SageMaker is a fully managed machine learning service provided by Amazon Web Services (AWS). It was launched in November 2017. Now, I will multiply the launch year by 2 using my calculator tool.</thinking> 
Tool #2: my_calculator_tool
<thinking>I have the result of the multiplication calculation. Now I can provide a complete answer to the user's question.</thinking>

Amaz

In [8]:
# Test with a calculation-only query
calc_query = "What is 156 multiplied by 23?"
print(f"Query: {calc_query}")
print(f"Response: {agent(calc_query)}")
print("\n" + "="*50 + "\n")

# Test with a Wikipedia-only query
wiki_query = "Tell me about artificial intelligence"
print(f"Query: {wiki_query}")
print(f"Response: {agent(wiki_query)}")
print("\n" + "="*50 + "\n")

# Test with a complex query requiring both tools
complex_query = "What is an Astronomical unit (AU)? Express the circumference of the earth in AU"
print(f"Query: {complex_query}")
print(f"Response: {agent(complex_query)}")

Query: What is 156 multiplied by 23?
<thinking>I need to multiply 156 by 23. I can use the calculator tool for this operation.</thinking> 
Tool #3: my_calculator_tool
The result of multiplying 156 by 23 is 3588.Response: The result of multiplying 156 by 23 is 3588.



Query: Tell me about artificial intelligence
<thinking>I need to provide an explanation of what artificial intelligence (AI) is. I'll use general knowledge to provide this information.</thinking>

Artificial intelligence (AI) is a field of computer science that focuses on creating intelligent machines which can work and react like humans, utilizing algorithms and statistical models to perform tasks. AI involves the development of algorithms that can perceive their environment and take actions that maximize their chances of successfully completing their goals. 

AI technologies include machine learning, natural language processing, and robotics, among others. AI is used in various fields, including healthcare, finance, and